In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")
import gp
import pickle
import matplotlib.pyplot as plt
import numpy as np
from pandas import DataFrame
import openpyxl
import math
# Making table for testing masses. Currently gives lots of inf values for unknown reason.
rebin = 40;
mass_hypothesis = .100
mass_hypotheses = np.linspace(.033,.179,147).tolist()
# Coefficients for the polynomial from the table for unsmeared (P0 to P4)
#coefficients = [0.00032, 0.019, -0.11, 1.39, -4.33]

#Smeared Coefficients
coefficients = [0.00038,0.041,-0.27,3.49,-11.11]


def sigma(mass, coeffs):
    #sigma is twice the mass resolution
    return sum(c * mass**i for i, c in enumerate(coeffs))

def sigmas(masses, coeffs):
    sigma_list = []
    for mass in masses:
        sigma_list.append(sum(c * mass**i for i, c in enumerate(coeffs)))
    return sigma_list
    
def make_table(masses, coeffs):
    sigma_list = sigmas(masses, coeffs)
    search_list = []
    p_list = []
    for i in range(len(masses)):
        # Print blind range for debugging
        blind_range = (masses[i] - sigma_list[i], masses[i] + sigma_list[i])
        print(f"Mass: {masses[i]}, Sigma: {sigma_list[i]}, Blind Range: {blind_range}")
        
        try:
            # Create the model with the current mass and blind range
            m = gp.GaussianProcessModel(
                h = ('EventSelection_Data_10Percent.root', 'h_Minv_General_Final_1'),
                kernel = gp.kernels.WhiteKernel(noise_level=7e3) + gp.kernels.RBF(length_scale = 0.016)* gp.kernels.DotProduct(sigma_0 = 2.5e4),
                blind_range = blind_range,
                #modify_histogram = [gp._hist.manipulation.rebin_and_limit(rebin, 0.033, 0.179), gp._hist.manipulation.inject_signal(5000, sigma_list[i], mass_hypothesis)]
                modify_histogram = gp._hist.manipulation.rebin_and_limit(rebin, 0.033, 0.179)


            )
            
            search, p_value = m.search_in_blind_region()
        except Exception as e:
            print(f"Error for mass {masses[i]}: {e}")
            upper_limit = float('inf')  # or a fallback value

        search_list.append(search)
        p_list.append(p_value)

    return search_list, sigma_list, p_list

search1, sl, pvl = make_table(mass_hypotheses, coefficients)
# Put table in Excel file

df = DataFrame({'Mass': mass_hypotheses, 'Blind Range Size': [i*2 for i in sl], 'Statistic': search1, 'p-value': pvl})
df.to_csv('search_test.csv', index=False)

Mass: 0.033, Sigma: 0.00155121454769, Blind Range: (0.03144878545231, 0.03455121454769)
Mass: 0.034, Sigma: 0.00158420426704, Blind Range: (0.03241579573296, 0.03558420426704)
Mass: 0.035, Sigma: 0.0016172118062500003, Blind Range: (0.033382788193750006, 0.03661721180625)
Mass: 0.036000000000000004, Sigma: 0.0016502489062400004, Blind Range: (0.034349751093760005, 0.037650248906240004)
Mass: 0.037000000000000005, Sigma: 0.0016833270412900004, Blind Range: (0.03531667295871001, 0.03868332704129)
Mass: 0.038, Sigma: 0.00171645741904, Blind Range: (0.03628354258096, 0.03971645741904)
Mass: 0.039, Sigma: 0.0017496509804900001, Blind Range: (0.03725034901951, 0.04074965098049)
Mass: 0.04, Sigma: 0.0017829184000000002, Blind Range: (0.0382170816, 0.0417829184)


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn

Mass: 0.041, Sigma: 0.0018162700852900001, Blind Range: (0.03918372991471, 0.04281627008529)
Mass: 0.042, Sigma: 0.0018497161774400001, Blind Range: (0.04015028382256, 0.043849716177440004)
Mass: 0.043000000000000003, Sigma: 0.0018832665508900003, Blind Range: (0.04111673344911, 0.044883266550890005)
Mass: 0.044, Sigma: 0.00191693081344, Blind Range: (0.04208306918656, 0.04591693081344)
Mass: 0.045, Sigma: 0.0019507183062500001, Blind Range: (0.04304928169375, 0.04695071830625)
Mass: 0.046, Sigma: 0.0019846381038400003, Blind Range: (0.044015361896159996, 0.04798463810384)
Mass: 0.047, Sigma: 0.00201869901409, Blind Range: (0.04498130098591, 0.04901869901409)
Mass: 0.048, Sigma: 0.00205290957824, Blind Range: (0.04594709042176, 0.050052909578240004)


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.wa

Mass: 0.049, Sigma: 0.00208727807089, Blind Range: (0.04691272192911, 0.05108727807089)
Mass: 0.05, Sigma: 0.0021218125, Blind Range: (0.0478781875, 0.0521218125)
Mass: 0.051000000000000004, Sigma: 0.0021565206068900004, Blind Range: (0.048843479393110005, 0.05315652060689)
Mass: 0.052000000000000005, Sigma: 0.00219140986624, Blind Range: (0.04980859013376, 0.054191409866240006)
Mass: 0.053000000000000005, Sigma: 0.0022264874860900004, Blind Range: (0.050773512513910005, 0.055226487486090006)
Mass: 0.054000000000000006, Sigma: 0.0022617604078400003, Blind Range: (0.05173823959216001, 0.056261760407840006)
Mass: 0.055, Sigma: 0.00229723530625, Blind Range: (0.05270276469375, 0.05729723530625)
Mass: 0.056, Sigma: 0.00233291858944, Blind Range: (0.05366708141056, 0.05833291858944)
Mass: 0.057, Sigma: 0.00236881639889, Blind Range: (0.05463118360111, 0.05936881639889)


...

/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.wa

Mass: 0.165, Sigma: 0.007237027306250001, Blind Range: (0.15776297269375, 0.17223702730625)
Mass: 0.166, Sigma: 0.007273981899040002, Blind Range: (0.15872601810096001, 0.17327398189904)
Mass: 0.167, Sigma: 0.007310198743689999, Blind Range: (0.15968980125631002, 0.17431019874369)
Mass: 0.168, Sigma: 0.007345654384640002, Blind Range: (0.16065434561536002, 0.17534565438464)
Mass: 0.169, Sigma: 0.007380325099690005, Blind Range: (0.16161967490031, 0.17638032509969)
Mass: 0.17, Sigma: 0.0074141869000000004, Blind Range: (0.1625858131, 0.17741418690000002)
Mass: 0.171, Sigma: 0.007447215530090004, Blind Range: (0.16355278446991, 0.17844721553009002)
Mass: 0.17200000000000001, Sigma: 0.007479386467840002, Blind Range: (0.16452061353216002, 0.17947938646784)
Mass: 0.17300000000000001, Sigma: 0.007510674924490001, Blind Range: (0.16548932507551, 0.18051067492449002)
Mass: 0.17400000000000002, Sigma: 0.0075410558446400025, Blind Range: (0.16645894415536, 0.18154105584464003)


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.wa

Mass: 0.17500000000000002, Sigma: 0.00757050390625, Blind Range: (0.16742949609375002, 0.18257050390625001)
Mass: 0.17600000000000002, Sigma: 0.0075989935206400025, Blind Range: (0.16840100647936002, 0.18359899352064002)
Mass: 0.17700000000000002, Sigma: 0.007626498832490003, Blind Range: (0.16937350116751002, 0.18462649883249002)
Mass: 0.178, Sigma: 0.007652993719840001, Blind Range: (0.17034700628016, 0.18565299371983998)
Mass: 0.179, Sigma: 0.007678451794090001, Blind Range: (0.17132154820591, 0.18667845179408998)


/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__sigma_0 is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__noise_level is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/aidanhsu/Documents/gaus-proc/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k1__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.wa